# Example 1.4.2 / 1: Range Constraints

Companion notebook for **1.4.2 Range and Logical Constraints** in the AMPL lab Documentation, section "Range Constraints". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Range Constraints

A diet must supply each nutrient between a minimum and a maximum. One declaration says both.
In `data.py` and `run.py`, `ampl` is an `AMPL` object that has already read `model.mod`.
After the solve, the `Constraint` object `ampl.con["Diet"]` reads the range back: its
`get_values(["lb", "body", "ub"])` returns the lower limit, the value of the middle expression
and the upper limit of every member, as a DataFrame:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set NUTR;
set FOOD;
param amt {NUTR, FOOD} >= 0;      # amount of nutrient per unit of food
param n_min {NUTR} >= 0;
param n_max {i in NUTR} >= n_min[i];
param cost {FOOD} > 0;
var Buy {FOOD} >= 0;

minimize Total_Cost: sum {j in FOOD} cost[j] * Buy[j];

subject to Diet {i in NUTR}:
    n_min[i] <= sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["NUTR"] = ["A", "C"]
ampl.set["FOOD"] = ["BEEF", "FISH"]
ampl.param["amt"] = {("A", "BEEF"): 60, ("A", "FISH"): 8,
                     ("C", "BEEF"): 20, ("C", "FISH"): 10}
ampl.param["n_min"] = {"A": 700, "C": 400}
ampl.param["n_max"] = {"A": 2000, "C": 2000}
ampl.param["cost"] = {"BEEF": 3.19, "FISH": 2.29}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.con["Diet"].get_values(["lb", "body", "ub"]).to_pandas())

HiGHS 1.14.0: optimal solution; objective 63.8
2 simplex iterations
0 barrier iterations
solved
   Diet.lb  Diet.body  Diet.ub
A      700       1200     2000
C      400        400     2000


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. The suffixes `.lb`, `.body` and `.ub` give
the same three values:

In [6]:
%%writefile data.dat
set NUTR := A C;
set FOOD := BEEF FISH;
param amt:  BEEF  FISH :=
  A          60     8
  C          20    10 ;
param:  n_min  n_max :=
  A      700   2000
  C      400   2000 ;
param cost := BEEF 3.19  FISH 2.29;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Diet.lb, Diet.body, Diet.ub;

HiGHS 1.14.0: optimal solution; objective 63.8
2 simplex iterations
0 barrier iterations
: Diet.lb Diet.body Diet.ub    :=
A    700     1200     2000
C    400      400     2000
;

